# Zadanie dodatkowe, część 1 (Colab): model CNN dla CIFAR-10 do wdrożenia w Databricks

Ten notebook przygotowuje dwa pliki, które w części 2 wgracie do Databricks Free Edition:

- `cifar_cnn.keras`, czyli wytrenowany model,
- `cifar_probka.npz`, czyli 200 obrazów testowych z etykietami i z predykcjami policzonymi tutaj, w TensorFlow.

Databricks Free Edition nie ma GPU, a jego środowisko serverless nie zawiera TensorFlow. Dlatego trenujemy w Colabie z GPU (Środowisko wykonawcze, Zmień typ środowiska wykonawczego, T4 GPU), a po stronie Databricks ten sam plik `.keras` wczytamy w Keras 3 z backendem PyTorch. Predykcje z próbki posłużą do sprawdzenia, czy model po przeniesieniu liczy to samo.

Jeśli macie zapisany model CNN z laboratorium 2 (CIFAR-10, wejście 32x32x3 w skali 0-255), możecie go użyć zamiast trenować nowy: w sekcji 3 ustawcie `UZYJ_WLASNEGO = True` i podajcie ścieżkę.

## 1. Konfiguracja

In [ ]:
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import numpy as np
import keras
from keras import layers

print("Keras", keras.__version__, "backend", keras.backend.backend())
keras.utils.set_random_seed(42)

KLASY = ["samolot", "samochód", "ptak", "kot", "jeleń", "pies", "żaba", "koń", "statek", "ciężarówka"]

## 2. Dane

In [ ]:
(x_tr, y_tr), (x_te, y_te) = keras.datasets.cifar10.load_data()
y_tr, y_te = y_tr.ravel(), y_te.ravel()
print(x_tr.shape, x_tr.dtype, x_te.shape)

## 3. Model

Normalizacja jest warstwą modelu (`Rescaling`), więc model przyjmuje surowe piksele 0-255 jako `float32`. To upraszcza wdrożenie: klient wysyła obraz w tej postaci, w jakiej go ma, i nie musi znać sposobu przetwarzania wstępnego.

Używamy wyłącznie standardowych warstw Keras. Warstwy pisane samodzielnie albo wywołania `tf.*` wewnątrz modelu nie przeniosłyby się na backend PyTorch.

In [ ]:
UZYJ_WLASNEGO = False
SCIEZKA_WLASNEGO = "lab02_cnn.keras"

def zbuduj_cnn():
    wej = keras.Input(shape=(32, 32, 3), name="obraz")
    x = layers.Rescaling(1 / 255)(wej)
    for filtry in (32, 64, 128):
        x = layers.Conv2D(filtry, 3, padding="same", activation="relu")(x)
        x = layers.Conv2D(filtry, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(0.2)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    wyj = layers.Dense(10, activation="softmax", name="klasa")(x)
    return keras.Model(wej, wyj, name="cifar_cnn")

if UZYJ_WLASNEGO:
    model = keras.saving.load_model(SCIEZKA_WLASNEGO)
else:
    model = zbuduj_cnn()
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

## 4. Trening

Liczba epok jest mała, bo celem zadania jest wdrożenie, a nie wynik. Na GPU w Colabie trening powinien zająć kilka minut, ale tego czasu nie mierzyliśmy. Jeśli macie mało czasu, zmniejszcie `EPOKI`.

In [ ]:
EPOKI = 15
if not UZYJ_WLASNEGO:
    historia = model.fit(
        x_tr.astype("float32"), y_tr,
        validation_split=0.1, epochs=EPOKI, batch_size=128,
        callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)],
        verbose=2,
    )
strata, dokl = model.evaluate(x_te.astype("float32"), y_te, verbose=0)
print(f"dokładność na zbiorze testowym: {dokl:.4f}")

## 5. Zapis modelu i próbki kontrolnej

Próbka to 200 losowych obrazów testowych (stałe ziarno). Zapisujemy też prawdopodobieństwa policzone w TensorFlow, żeby w Databricks porównać je z wynikiem backendu PyTorch i z odpowiedzią endpointu.

In [ ]:
model.save("cifar_cnn.keras")

rng = np.random.default_rng(0)
idx = rng.choice(len(x_te), size=200, replace=False)
x_probka = x_te[idx]
y_probka = y_te[idx]
p_probka = model.predict(x_probka.astype("float32"), verbose=0)

np.savez_compressed("cifar_probka.npz", x=x_probka, y=y_probka, p_tf=p_probka,
                    klasy=np.array(KLASY))
print("dokładność na próbce:", (p_probka.argmax(1) == y_probka).mean())
print({f: round(os.path.getsize(f) / 1e6, 2) for f in ["cifar_cnn.keras", "cifar_probka.npz"]}, "MB")

## 6. Sprawdzenie: czy zapisany model wczytuje się i liczy to samo

In [ ]:
m2 = keras.saving.load_model("cifar_cnn.keras")
roznica = np.abs(m2.predict(x_probka.astype("float32"), verbose=0) - p_probka).max()
print("maksymalna różnica prawdopodobieństw:", roznica)
assert roznica < 1e-5

## 7. Pobranie plików

W Colabie poniższa komórka pobierze oba pliki na Wasz komputer. Stąd wgracie je do Volume w Databricks (instrukcja, krok 3).

In [ ]:
try:
    from google.colab import files
    files.download("cifar_cnn.keras")
    files.download("cifar_probka.npz")
except ImportError:
    print("Poza Colabem: pliki są w bieżącym katalogu:", os.getcwd())